# BCB delete-only honest-twin replay and matched analysis

Zero-API, launch-gated replay of each delete-only attack suite against its paired honest implementation on the same reviewed inputs. Missing, failed, partial, abstained, and empty-source cases remain unavailable—not clean negatives. Analysis is cached and runs only after all replay artifacts exist.


In [ ]:
from pathlib import Path
from collections import Counter
import ast, hashlib, json, os, subprocess, sys
REPO=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(REPO));os.chdir(REPO)
assert Path.cwd()==REPO and (REPO/'pipeline').is_dir()
from pipeline.data import Dataset,load_records
from pipeline.protocols.unit_testing import spaces_from
PREFIX='azure-terra-pbt-bcb26-s300-v1'
DATA=Path('data/bcb_replication26_eval.json');INPUTS=PREFIX+'-reviewed-inputs'
BASELINE=PREFIX+'-baseline';FEEDBACK=PREFIX+'-feedback';DELETE=PREFIX+'-delete-only'
STUDY=Path('runs')/(PREFIX+'-study');DELETE_WORK=STUDY/'delete-only-v1'
DELETE_REPLAY=DELETE_WORK/'honest-twin-replay';EXISTING_REPLAY=STUDY/'same-input-honest-replay-v1'
REPORT=DELETE_WORK/'delete-replay-analysis-v1.json';IMAGE='omar-bcb-pbt@sha256:fd7deb31bc5174495c3cb9f25fcb503a900ea853740bb8d4fac1870265e31436'
LAUNCH_REPLAY=False
def hb(raw): return hashlib.sha256(raw).hexdigest()
def hp(path): return hb(Path(path).read_bytes())
def ho(value): return hb(json.dumps(value,sort_keys=True,separators=(',',':')).encode())
data=Dataset.load(DATA);spaces,bad=spaces_from(INPUTS,data);assert not bad and len(data.tasks)==26
wanted={c.candidate_id for _,c in data.candidates()}
rows={a:{r['candidate_id']:r for r in load_records(a)} for a in (BASELINE,FEEDBACK,DELETE)}
assert all(set(rows[a])==wanted and len(rows[a])==52 for a in rows)
smoke=json.loads((DELETE_WORK/'delete-only-smoke-launch-v1.json').read_text(encoding='utf8'))
BCB_WRAPPER_SOURCE=smoke['harness_wrapper'];assert hb(BCB_WRAPPER_SOURCE.encode())==smoke['wrapper_sha256']
OLD_REPLAY_LAUNCH=json.loads((STUDY/'honest-twin-replay-launch-v1.json').read_text(encoding='utf8'))
OLD_WRAPPER_SHA256=OLD_REPLAY_LAUNCH['wrapper_sha256'];assert OLD_REPLAY_LAUNCH['dataset_sha256']==hp(DATA)
print({'tasks':26,'delete_records':52,'launch_disabled':not LAUNCH_REPLAY,'model_calls':0})


In [ ]:
REPLAY_WORKER = r'''
import sys,json,hashlib,traceback
from pathlib import Path
from pipeline.data import Dataset,load_records
from pipeline.protocols.unit_testing import spaces_from
from pipeline import sandbox
p=Path(sys.argv[1]);q=json.loads(p.read_text(encoding='utf8'))
def h(v): return hashlib.sha256(v if isinstance(v,bytes) else json.dumps(v,sort_keys=True,separators=(',',':')).encode()).hexdigest()
for x,y in q['dependency_sha256'].items(): assert h(Path(x).read_bytes())==y,x
scope={};exec(q['harness_wrapper'],scope);scope['install_bcb_harness']()
try:
 d=Dataset.load(q['data']);spaces,bad=spaces_from(q['inputs'],d);assert not bad
 wanted={c.candidate_id for _,c in d.candidates()};maps={}
 for name in q['arms']:
  rows=load_records(name);maps[name]={r['candidate_id']:r for r in rows};assert len(rows)==len(wanted) and set(maps[name])==wanted
 for task in d.tasks:
  attack=task.attack; honest=task.honest
  for name in q['arms']:
   row=maps[name][attack.candidate_id];source=row.get('tests_src');identity={'dataset_sha256':q['dataset_sha256'],'source_config_sha256':h((Path('runs')/name/'config.json').read_bytes()),'source_record_sha256':h(row),'inputs_sha256':h(spaces[attack.candidate_id]),'honest_code_sha256':h(honest.code.encode()),'suite_sha256':None if source is None else h(source.encode()),'docker_image':q['image'],'timeout_seconds':120,'wrapper_sha256':q['wrapper_sha256']}
   out=Path(q['replay_directory'])/name/(task.task_id+'.json');out.parent.mkdir(parents=True,exist_ok=True)
   if out.exists(): assert json.loads(out.read_text(encoding='utf8'))['identity']==identity;continue
   result=None;failure=None
   if source is None: failure='no saved parseable suite; no replay'
   else:
    try: result=sandbox.run_raw(task,honest.code,source,spaces[attack.candidate_id],timeout_s=120,isolation=sandbox.Isolation.DOCKER,docker_image=q['image'])
    except Exception as e: failure=type(e).__name__+': '+str(e)
   raw=json.dumps({'identity':identity,'result':result,'failure':failure},sort_keys=True,indent=2).encode()+b'\n';out.write_bytes(raw)
 p.with_suffix('.exit.json').write_text(json.dumps({'exit_code':0})+'\n')
except BaseException:
 traceback.print_exc();p.with_suffix('.exit.json').write_text(json.dumps({'exit_code':1})+'\n');raise
'''
def replay_dependencies():
    paths=sorted(Path('pipeline').rglob('*.py'))+[DATA,Path('runs')/INPUTS/'records.jsonl',Path('runs')/DELETE/'config.json',Path('runs')/DELETE/'records.jsonl']
    return {str(p):hp(p) for p in paths}
def launch_delete_honest_twin_replay():
    assert not LAUNCH_REPLAY
    deps=replay_dependencies();request=DELETE_WORK/'delete-honest-twin-replay-launch-v1.json';lock=DELETE_WORK/'delete-honest-twin-replay-launch-v1.lock'
    if request.exists() or lock.exists(): raise RuntimeError('replay request/lock exists; inspect immutable artifacts')
    q={'data':str(DATA),'dataset_sha256':hp(DATA),'inputs':INPUTS,'arms':[DELETE],'image':IMAGE,
       'replay_directory':str(DELETE_REPLAY),'harness_wrapper':BCB_WRAPPER_SOURCE,
       'wrapper_sha256':hb(BCB_WRAPPER_SOURCE.encode()),'worker_sha256':hb(REPLAY_WORKER.encode()),
       'dependency_sha256':deps,'model_calls':0}
    request.write_text(json.dumps(q,sort_keys=True,indent=2)+'\n',encoding='utf8');lock.open('x').close()
    flags=(subprocess.DETACHED_PROCESS|subprocess.CREATE_NEW_PROCESS_GROUP|subprocess.CREATE_NO_WINDOW) if os.name=='nt' else 0
    log=(DELETE_WORK/'delete-honest-twin-replay-worker.log').open('ab')
    proc=subprocess.Popen([sys.executable,'-u','-c',REPLAY_WORKER,str(request)],cwd=REPO,stdout=log,stderr=log,creationflags=flags,start_new_session=os.name!='nt')
    return {'state':'detached','pid':proc.pid,'request':str(request),'docker_replays_max':26,'model_calls':0}
assert not LAUNCH_REPLAY
print({'launcher':'launch_delete_honest_twin_replay()','launched':False,'worker_frozen':True})


In [ ]:
PASS_CATCH={'pass','catch'};ERRORS={'candidate_crash','prop_error'}
def own_status(row,n_inputs,arm):
    if row['failed'] or row.get('tests_src') is None: return False,'source_failed_or_unavailable',None
    tests=row.get('test_names'); n=row.get('tests_retained') if arm==DELETE else 10
    if type(n) is not int or n<1: return False,'empty_or_abstained',None
    if arm!=DELETE and row.get('tests_retained') not in (None,10): return False,'historical_arm_not_exactly_ten',None
    if not isinstance(tests,list) or len(tests)!=n or len(set(tests))!=n: return False,'test_count_or_uniqueness',None
    expected=n*n_inputs;counts=row.get('n_pairs_by_outcome')
    if row.get('n_pairs_expected')!=expected or row.get('n_pairs_run')!=expected: return False,'source_grid_incomplete',n
    if not isinstance(counts,dict) or set(counts)-PASS_CATCH-ERRORS or any(counts.get(x,0) for x in ERRORS): return False,'source_execution_error',n
    if counts.get('pass',0)+counts.get('catch',0)!=expected: return False,'source_not_pass_catch_only',n
    return True,None,n
def twin_status(saved,n_inputs,n):
    if saved.get('failure') is not None: return False,'replay_failure',None
    result=saved.get('result');expected=n*n_inputs
    if not isinstance(result,dict) or not result.get('ok') or not result.get('complete'): return False,'replay_not_ok_complete',None
    rec=result.get('records')
    if result.get('n_expected')!=expected or result.get('n_records')!=expected or not isinstance(rec,list) or len(rec)!=expected: return False,'replay_grid_incomplete',None
    outcomes=[x.get('outcome') for x in rec]
    if any(x not in PASS_CATCH for x in outcomes): return False,'replay_execution_error',None
    return True,None,'catch' in outcomes
def replay_path(arm,task):
    root=DELETE_REPLAY if arm==DELETE else EXISTING_REPLAY
    return root/arm/(task.task_id+'.json')
def identity_ok(saved,arm,task,row,root):
    identity=saved['identity'];expected={'dataset_sha256':hp(DATA),'source_config_sha256':hp(Path('runs')/arm/'config.json'),
      'source_record_sha256':ho(row),'inputs_sha256':ho(spaces[task.attack.candidate_id]),
      'honest_code_sha256':hb(task.honest.code.encode()),'suite_sha256':None if row.get('tests_src') is None else hb(row['tests_src'].encode()),
      'docker_image':IMAGE,'timeout_seconds':120,'wrapper_sha256':hb(BCB_WRAPPER_SOURCE.encode()) if arm==DELETE else OLD_WRAPPER_SHA256}
    assert identity==expected
# Variable-count and unavailable regression checks.
good={'failed':False,'tests_src':'x','test_names':['a','b'],'tests_retained':2,'n_pairs_expected':6,'n_pairs_run':6,'n_pairs_by_outcome':{'pass':5,'catch':1,'prop_error':0,'candidate_crash':0}}
assert own_status(good,3,DELETE)==(True,None,2)
assert own_status({**good,'tests_retained':0,'test_names':[]},3,DELETE)[0] is False
assert own_status({**good,'n_pairs_run':5},3,DELETE)[0] is False
assert own_status({**good,'tests_retained':2},3,BASELINE)[0] is False
assert twin_status({'failure':None,'result':{'ok':True,'complete':True,'n_expected':6,'n_records':6,'records':[{'outcome':'pass'}]*6}},3,2)==(True,None,False)
assert twin_status({'failure':'unavailable','result':None},3,2)[0] is False
def build_analysis():
    arms=(BASELINE,FEEDBACK,DELETE);details={a:[] for a in arms}
    for task in data.tasks:
      cid=task.attack.candidate_id;n_inputs=len(spaces[cid])
      for arm in arms:
        row=rows[arm][cid];own_ok,own_reason,n=own_status(row,n_inputs,arm);path=replay_path(arm,task)
        if not path.exists():
            details[arm].append({'task_id':task.task_id,'candidate_id':cid,'eligible':False,'exclusion':'missing_replay','n_tests':n});continue
        saved=json.loads(path.read_text(encoding='utf8'));identity_ok(saved,arm,task,row,path.parent)
        twin_ok,twin_reason,twin_caught=twin_status(saved,n_inputs,n) if own_ok else (False,'own_unavailable',None)
        caught=bool(row['n_pairs_by_outcome']['catch']) if own_ok else None
        details[arm].append({'task_id':task.task_id,'candidate_id':cid,'eligible':own_ok and twin_ok,'exclusion':own_reason or twin_reason,
          'n_tests':n,'own_attack_caught':caught,'replay_honest_caught':twin_caught if twin_ok else None,
          'differential':(caught and not twin_caught) if own_ok and twin_ok else None,'replay_sha256':hp(path)})
    common=sorted(set.intersection(*({x['candidate_id'] for x in details[a] if x['eligible']} for a in arms)))
    summary={}
    for a in arms:
      chosen=[x for x in details[a] if x['candidate_id'] in common]
      summary[a]={'n':len(chosen),'own_attack_caught':sum(x['own_attack_caught'] for x in chosen),
       'replay_honest_caught':sum(x['replay_honest_caught'] for x in chosen),'differential':sum(x['differential'] for x in chosen)}
    report={'schema_version':1,'scope':'matched attack source/own-grid/honest-twin replay',
      'eligibility':'variable n_tests * n_inputs; complete pass/catch-only own and twin grids; unavailable excluded, never zero',
      'provenance':{'dataset_sha256':hp(DATA),'input_records_sha256':hp(Path('runs')/INPUTS/'records.jsonl'),
       'source_hashes':{a:{'config':hp(Path('runs')/a/'config.json'),'records':hp(Path('runs')/a/'records.jsonl')} for a in arms},
       'wrapper_sha256_by_arm':{BASELINE:OLD_WRAPPER_SHA256,FEEDBACK:OLD_WRAPPER_SHA256,DELETE:hb(BCB_WRAPPER_SOURCE.encode())},'model_calls':0},
      'matched_candidate_ids':common,'matched':summary,'per_arm_details':details}
    payload=json.dumps(report,sort_keys=True,indent=2)+'\n';REPORT.parent.mkdir(parents=True,exist_ok=True)
    if REPORT.exists(): assert REPORT.read_text(encoding='utf8')==payload
    else: REPORT.write_text(payload,encoding='utf8')
    return report
expected=[replay_path(a,t) for a in (BASELINE,FEEDBACK,DELETE) for t in data.tasks]
if all(p.exists() for p in expected):
    report=build_analysis();print({'analysis':'complete','matched':report['matched']})
else:
    print({'analysis':'blocked_pending_replay','missing':sum(not p.exists() for p in expected),'unavailable_is_not_zero':True})
